# T35 - 不同曲线投资顺序分析
## 第2章：周期标注

### 概述
本章生成收益率下行周期的标注数据，用于后续的策略分析。包括：
1. 定义历史收益率下行周期
2. 生成周期标注DataFrame
3. 可视化周期分布
4. 保存周期标注文件

In [ ]:
# 1. 导入必要的库
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import os
import warnings

warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("库导入成功！")

In [ ]:
# 2. 加载配置
from config import (
    YIELD_DOWN_CYCLES,
    CYCLE_LABEL_FILE,
    OUTPUT_DIR,
    CHART_COLORS,
    FIGURE_SIZE
)

print(f"输出文件: {CYCLE_LABEL_FILE}")
print(f"周期数量: {len(YIELD_DOWN_CYCLES)}")

In [ ]:
# 3. 创建周期标注DataFrame
def create_cycle_labels():
    """创建周期标注数据"""
    # 从配置创建DataFrame
    cycles_df = pd.DataFrame(YIELD_DOWN_CYCLES)
    
    # 重命名列以符合命名规范
    cycles_df.columns = ['cycle', 'start_date', 'end_date', 'start_yield', 'end_yield', 'yield_change']
    
    # 计算周期天数
    cycles_df['start_date'] = pd.to_datetime(cycles_df['start_date'])
    cycles_df['end_date'] = pd.to_datetime(cycles_df['end_date'])
    cycles_df['duration_days'] = (cycles_df['end_date'] - cycles_df['start_date']).dt.days
    
    return cycles_df

# 创建周期标注
cycles_df = create_cycle_labels()

print("周期标注数据:")
print("=" * 80)
print(cycles_df.to_string(index=False))

In [ ]:
# 4. 周期统计分析
print("周期统计分析")
print("=" * 60)

print("\n1. 基本统计:")
print(f"   总周期数: {len(cycles_df)}")
print(f"   时间跨度: {cycles_df['start_date'].min().strftime('%Y-%m-%d')} ~ {cycles_df['end_date'].max().strftime('%Y-%m-%d')}")

print("\n2. 收益率变动统计:")
print(f"   平均变动: {cycles_df['yield_change'].mean():.4f}%")
print(f"   最大变动: {cycles_df['yield_change'].min():.4f}%")
print(f"   最小变动: {cycles_df['yield_change'].max():.4f}%")
print(f"   标准差: {cycles_df['yield_change'].std():.4f}%")

print("\n3. 周期长度统计:")
print(f"   平均长度: {cycles_df['duration_days'].mean():.0f} 天")
print(f"   最短周期: {cycles_df['duration_days'].min()} 天")
print(f"   最长周期: {cycles_df['duration_days'].max()} 天")

print("\n4. 各周期详情:")
for _, row in cycles_df.iterrows():
    print(f"\n   周期 {row['cycle']}:")
    print(f"      时间: {row['start_date'].strftime('%Y-%m-%d')} ~ {row['end_date'].strftime('%Y-%m-%d')}")
    print(f"      持续: {row['duration_days']} 天 ({row['duration_days']/365:.1f} 年)")
    print(f"      收益率: {row['start_yield']:.2f}% → {row['end_yield']:.2f}% (变动 {row['yield_change']:.2f}%)")

In [ ]:
# 5. 生成每个周期的交易日序列
def generate_cycle_trading_days():
    """生成每个周期的交易日序列"""
    all_trading_days = []
    
    for _, cycle in cycles_df.iterrows():
        # 生成工作日序列（周一到周五）
        dates = pd.date_range(
            start=cycle['start_date'],
            end=cycle['end_date'],
            freq='B'  # 工作日
        )
        
        for date in dates:
            all_trading_days.append({
                'dt': date,
                'cycle_id': cycle['cycle']
            })
    
    return pd.DataFrame(all_trading_days)

# 生成交易日序列
trading_days_df = generate_cycle_trading_days()

print(f"\n交易日序列统计:")
print(f"   总交易日数: {len(trading_days_df)}")
print(f"\n各周期交易日数:")
for cycle_id in sorted(trading_days_df['cycle_id'].unique()):
    count = len(trading_days_df[trading_days_df['cycle_id'] == cycle_id])
    print(f"   周期 {cycle_id}: {count} 个交易日")

In [ ]:
# 6. 可视化周期分布
def plot_cycle_distribution(cycles_df):
    """绘制周期分布图"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 设置中文字体
    plt.rcParams['axes.unicode_minus'] = False
    
    # 图1：收益率变动柱状图
    ax1 = axes[0, 0]
    colors = [CHART_COLORS[i % len(CHART_COLORS)] for i in range(len(cycles_df))]
    bars = ax1.bar(cycles_df['cycle'], cycles_df['yield_change'], color=colors)
    ax1.set_xlabel('周期编号')
    ax1.set_ylabel('收益率变动 (%)')
    ax1.set_title('各周期收益率变动')
    ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax1.grid(True, alpha=0.3)
    
    # 添加数值标签
    for bar, value in zip(bars, cycles_df['yield_change']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.1,
                f'{value:.2f}%', ha='center', va='top', fontsize=9)
    
    # 图2：周期长度柱状图
    ax2 = axes[0, 1]
    bars = ax2.bar(cycles_df['cycle'], cycles_df['duration_days'], color=colors)
    ax2.set_xlabel('周期编号')
    ax2.set_ylabel('持续天数')
    ax2.set_title('各周期持续天数')
    ax2.grid(True, alpha=0.3)
    
    # 添加数值标签
    for bar, value in zip(bars, cycles_df['duration_days']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{value}天', ha='center', va='bottom', fontsize=9)
    
    # 图3：起止收益率对比
    ax3 = axes[1, 0]
    x = np.arange(len(cycles_df))
    width = 0.35
    ax3.bar(x - width/2, cycles_df['start_yield'], width, label='起始收益率', color=CHART_COLORS[0])
    ax3.bar(x + width/2, cycles_df['end_yield'], width, label='结束收益率', color=CHART_COLORS[2])
    ax3.set_xlabel('周期编号')
    ax3.set_ylabel('收益率 (%)')
    ax3.set_title('各周期起止收益率对比')
    ax3.set_xticks(x)
    ax3.set_xticklabels(cycles_df['cycle'])
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 图4：时间线图
    ax4 = axes[1, 1]
    for i, (_, row) in enumerate(cycles_df.iterrows()):
        ax4.barh(i, row['duration_days'], left=row['start_date'], 
                height=0.5, color=CHART_COLORS[i % len(CHART_COLORS)],
                label=f"周期{row['cycle']}")
    ax4.set_xlabel('日期')
    ax4.set_ylabel('周期')
    ax4.set_title('周期时间分布')
    ax4.set_yticks(range(len(cycles_df)))
    ax4.set_yticklabels([f"周期{c}" for c in cycles_df['cycle']])
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # 保存图表
    output_path = os.path.join(OUTPUT_DIR, '周期分布图.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"图表已保存: {output_path}")
    
    plt.show()

# 绘制周期分布图
plot_cycle_distribution(cycles_df)

In [ ]:
# 7. 保存周期标注文件
def save_cycle_labels(cycles_df, trading_days_df):
    """保存周期标注文件"""
    # 保存周期汇总
    cycle_summary = cycles_df[['cycle', 'start_date', 'end_date', 'start_yield', 'end_yield', 'yield_change']].copy()
    cycle_summary['start_date'] = cycle_summary['start_date'].dt.strftime('%Y-%m-%d')
    cycle_summary['end_date'] = cycle_summary['end_date'].dt.strftime('%Y-%m-%d')
    
    cycle_summary.to_csv(CYCLE_LABEL_FILE, index=False, encoding='utf-8-sig', float_format='%.4f')
    print(f"周期标注文件已保存: {CYCLE_LABEL_FILE}")
    
    # 保存交易日序列
    trading_days_file = os.path.join(OUTPUT_DIR, '交易日序列.csv')
    trading_days_df['dt'] = trading_days_df['dt'].dt.strftime('%Y-%m-%d')
    trading_days_df.to_csv(trading_days_file, index=False, encoding='utf-8-sig')
    print(f"交易日序列已保存: {trading_days_file}")

# 保存文件
save_cycle_labels(cycles_df, trading_days_df)

In [ ]:
# 8. 验证保存的文件
def verify_saved_files():
    """验证保存的文件"""
    print("验证保存的文件:")
    print("=" * 60)
    
    # 验证周期标注文件
    if os.path.exists(CYCLE_LABEL_FILE):
        df = pd.read_csv(CYCLE_LABEL_FILE, encoding='utf-8-sig')
        print(f"\n1. 周期标注文件 ({os.path.basename(CYCLE_LABEL_FILE)}):")
        print(f"   记录数: {len(df)}")
        print(f"   列名: {list(df.columns)}")
        print(f"   文件大小: {os.path.getsize(CYCLE_LABEL_FILE):,} bytes")
        print("\n   内容预览:")
        print(df.to_string(index=False))
    else:
        print(f"错误: 文件不存在 - {CYCLE_LABEL_FILE}")
    
    # 验证交易日序列文件
    trading_days_file = os.path.join(OUTPUT_DIR, '交易日序列.csv')
    if os.path.exists(trading_days_file):
        df = pd.read_csv(trading_days_file, encoding='utf-8-sig')
        print(f"\n2. 交易日序列文件 ({os.path.basename(trading_days_file)}):")
        print(f"   记录数: {len(df)}")
        print(f"   列名: {list(df.columns)}")
        print(f"   文件大小: {os.path.getsize(trading_days_file):,} bytes")
        print("\n   前5条记录:")
        print(df.head().to_string(index=False))
    else:
        print(f"错误: 文件不存在 - {trading_days_file}")

verify_saved_files()

### 总结

本章完成了以下工作：
1. 创建了6个历史收益率下行周期的标注数据
2. 生成了每个周期的交易日序列
3. 可视化了周期分布（收益率变动、持续天数、时间线）
4. 保存了周期标注文件和交易日序列文件

**输出文件:**
- `收益率下行周期标注.csv`: 周期汇总信息
- `交易日序列.csv`: 每个周期的交易日序列
- `周期分布图.png`: 周期可视化图表

**下一步**: 运行 `03-数据获取.ipynb` 从数据库获取债券收益率数据。